# Ethical Web Scraper Architecture

## 1. Project Overview
**Goal**: To build a modular, reusable Python web scraper that strictly adheres to data privacy and ethical boundaries.

This framework is specifically designed for sensitive applications, such as scraping **support group meetings** or **grant data**. It prioritizes:
- **Politeness**: Avoiding server overload by honoring rate limits.
- **Compliance**: Automatically parsing and following `robots.txt` directives.
- **Extensibility**: Utilizing Object-Oriented patterns to easily adapt to new target sites without duplicating core logic.

## 2. Configuration Layer
Defines the global settings such as Custom User-Agent strings, rate limits, target URLs, and now the **Database Engine Configuration**.

In [ ]:
import urllib.robotparser
import urllib.parse
import urllib.request
import time
from abc import ABC, abstractmethod
from typing import Optional, Dict, Any

# Global Configuration Layer
CONFIG = {
    # Scraper Identity
    'USER_AGENT': 'EthicalDataBot/1.0 (+mailto:research@example.org)',
    'DEFAULT_CRAWL_DELAY': 3.0, 
    'TARGET_URLS': [
        'https://en.wikipedia.org/wiki/Main_Page',
    ],
    
    # Data Storage Configuration
    # Toggle 'DB_TYPE' between 'postgres', 'mongodb', or 'mock' to seamlessly swap engines.
    'DB_TYPE': 'mock', 
    'DB_CONNECTION_STRING': 'mock://localhost:1234'
}


## 3. The Ethical Compliance Class
A standalone object responsible for ensuring the scraper never violates host guidelines. It caches `robots.txt` files per domain and calculates required sleep intervals.

In [ ]:
class EthicalComplianceManager:
    """
    Dynamically checks robots.txt permissions and enforces crawl delays.
    Caches the parser instances to prevent redundant requests to the host.
    """
    def __init__(self, user_agent: str):
        self.user_agent = user_agent
        self._parsers: Dict[str, urllib.robotparser.RobotFileParser] = {}
        self._last_request_time: Dict[str, float] = {}

    def _get_domain_root(self, url: str) -> str:
        parsed = urllib.parse.urlparse(url)
        return f"{parsed.scheme}://{parsed.netloc}"

    def _get_parser(self, url: str) -> urllib.robotparser.RobotFileParser:
        domain = self._get_domain_root(url)
        if domain not in self._parsers:
            rp = urllib.robotparser.RobotFileParser()
            robots_url = urllib.parse.urljoin(domain, '/robots.txt')
            rp.set_url(robots_url)
            try:
                req = urllib.request.Request(robots_url, headers={'User-Agent': self.user_agent})
                with urllib.request.urlopen(req, timeout=10) as response:
                    rp.parse(response.read().decode('utf-8').splitlines())
            except Exception as e:
                print(f"[!] Warning: Could not parse robots.txt for {domain}. Assuming restrictive defaults. ({e})")
            
            self._parsers[domain] = rp
            self._last_request_time[domain] = 0.0
        return self._parsers[domain]

    def can_fetch(self, url: str) -> bool:
        rp = self._get_parser(url)
        return rp.can_fetch(self.user_agent, url)

    def get_crawl_delay(self, url: str, fallback_delay: float) -> float:
        rp = self._get_parser(url)
        delay = rp.crawl_delay(self.user_agent)
        if delay is None:
            delay = rp.crawl_delay('*')
        return delay if delay is not None else fallback_delay

    def apply_rate_limit(self, url: str, fallback_delay: float):
        domain = self._get_domain_root(url)
        delay = self.get_crawl_delay(url, fallback_delay)
        self._get_parser(url)
        elapsed = time.time() - self._last_request_time.get(domain, 0.0)
        if elapsed < delay:
            sleep_time = delay - elapsed
            print(f"[*] Polite backoff: Sleeping for {sleep_time:.2f}s to respect {domain} rate limits.")
            time.sleep(sleep_time)
            
    def mark_request_complete(self, url: str):
        domain = self._get_domain_root(url)
        self._last_request_time[domain] = time.time()


## 4. The Data Storage Strategy Layer
Implements the **Strategy Pattern** to decouple the core scraping algorithm from specific database mechanisms. By abstracting the storage layer, concrete scrapers remain pure (they only fetch and parse). The `DatabaseFactory` resolves the actual engine at runtime, protecting scraping logic from database migration changes.

In [ ]:
class BaseDatabase(ABC):
    """
    Abstract Strategy establishing the strict contract for data persistence.
    """
    def __init__(self, connection_string: str):
        self.connection_string = connection_string
        self.connect()

    @abstractmethod
    def connect(self):
        """Initialize the database connection pool or client."""
        pass

    @abstractmethod
    def save(self, collection_name: str, data: Dict[str, Any]):
        """Persist the parsed data entity into the target database."""
        pass


class PostgreSQLDatabase(BaseDatabase):
    """Concrete Strategy for Relational Data Storage (SQL)."""
    def connect(self):
        print(f"[DB - PostgreSQL] Initializing connection to {self.connection_string}...")
        # e.g., self.conn = psycopg2.connect(self.connection_string)
        
    def save(self, collection_name: str, data: Dict[str, Any]):
        print(f"[DB - PostgreSQL] Translating dictionary to relational INSERT for table '{collection_name}'...")
        # e.g., cursor = self.conn.cursor()
        # keys = ', '.join(data.keys())
        # values = ', '.join(['%s'] * len(data))
        # sql = f"INSERT INTO {collection_name} ({keys}) VALUES ({values})"
        # cursor.execute(sql, tuple(data.values()))
        print("  -> Execution: INSERT INTO complete.")


class MongoDBDatabase(BaseDatabase):
    """Concrete Strategy for Document Data Storage (NoSQL)."""
    def connect(self):
        print(f"[DB - MongoDB] Initializing connection to {self.connection_string}...")
        # e.g., self.client = pymongo.MongoClient(self.connection_string)
        
    def save(self, collection_name: str, data: Dict[str, Any]):
        print(f"[DB - MongoDB] Inserting flexible BSON document into collection '{collection_name}'...")
        # e.g., db = self.client.get_database()
        # db[collection_name].insert_one(data)
        print("  -> Execution: insert_one() complete.")


class MockDatabase(BaseDatabase):
    """A safe mock strategy for testing dry runs without requiring a real database."""
    def connect(self):
        print(f"[DB - Mock] Initialized Mock Database Strategy.")
        
    def save(self, collection_name: str, data: Dict[str, Any]):
        print(f"[DB - Mock] Received payload for '{collection_name}'. Document: '{data.get('page_title', 'Unknown')}'")


class DatabaseFactory:
    """
    Instantiates the correct database engine dynamically.
    """
    @staticmethod
    def get_database(db_type: str, connection_string: str) -> BaseDatabase:
        strategies = {
            'postgres': PostgreSQLDatabase,
            'mongodb': MongoDBDatabase,
            'mock': MockDatabase
        }
        db_class = strategies.get(db_type.lower())
        if not db_class:
            raise ValueError(f"Unsupported DB Type: {db_type}")
        return db_class(connection_string)


## 5. The Modular Scraper Base Class & Factory
Follows the **Factory Pattern** and **Template Method Pattern**. 
The base class handles all standard compliant HTTP requests. It now requires a `BaseDatabase` injection so it can automatically route parsed data to the persistence layer, hiding DB complexity from specific site parsers.

In [ ]:
class BaseScraper(ABC):
    """
    Abstract base class establishing the contract for all domain-specific scrapers.
    By injecting a database dependency here, concrete scrapers (like WikipediaScraper)
    never need to know about SQL or NoSQL syntax, fulfilling the Single Responsibility Principle.
    """
    def __init__(self, compliance_manager: EthicalComplianceManager, db_handler: BaseDatabase):
        self.compliance = compliance_manager
        self.db = db_handler

    def fetch_content(self, url: str) -> Optional[str]:
        if not self.compliance.can_fetch(url):
            print(f"[X] Blocked by robots.txt: Permission denied for {url}")
            return None
            
        self.compliance.apply_rate_limit(url, CONFIG['DEFAULT_CRAWL_DELAY'])
        
        print(f"[>] Fetching: {url}")
        try:
            req = urllib.request.Request(url, headers={'User-Agent': self.compliance.user_agent})
            with urllib.request.urlopen(req, timeout=15) as response:
                html = response.read().decode('utf-8')
                
            self.compliance.mark_request_complete(url)
            return html
            
        except Exception as e:
            print(f"[!] Failed to fetch {url}. Error: {e}")
            self.compliance.mark_request_complete(url)
            return None

    @abstractmethod
    def parse(self, html: str) -> Dict[str, Any]:
        """Extract entities from raw HTML. Implementations must return a clean dictionary."""
        pass

    def scrape(self, url: str, destination: str) -> Any:
        """
        Template method connecting the compliant fetch, domain-specific parse, 
        and automated DB storage mechanisms.
        """
        html = self.fetch_content(url)
        if html:
            # 1. Parse the specific domain logic
            parsed_data = self.parse(html)
            
            # 2. Delegate to the Strategy Pattern database engine to handle persistence automatically
            if parsed_data:
                self.db.save(destination, parsed_data)
                
            return parsed_data
        return None


class ScraperFactory:
    """
    Instantiates the correct scraper subclass based on the target URL's domain.
    """
    _registry = {}

    @classmethod
    def register(cls, domain: str):
        def inner_wrapper(wrapped_class):
            cls._registry[domain] = wrapped_class
            return wrapped_class
        return inner_wrapper

    @classmethod
    def get_scraper(cls, url: str, compliance_manager: EthicalComplianceManager, db_handler: BaseDatabase) -> BaseScraper:
        domain = urllib.parse.urlparse(url).netloc
        scraper_cls = cls._registry.get(domain)
        if not scraper_cls:
            raise NotImplementedError(f"No scraper implemented for domain: {domain}")
        return scraper_cls(compliance_manager, db_handler)


### 5.1 Specialized Scraper Implementation
An example parser for Wikipedia. Notice how it knows **absolutely nothing** about PostgreSQL or MongoDB. It only extracts data.

In [ ]:
@ScraperFactory.register('en.wikipedia.org')
class WikipediaScraper(BaseScraper):
    """
    Concrete implementation tailored for Wikipedia's DOM structure.
    """
    def parse(self, html: str) -> Dict[str, Any]:
        # Lightweight text extraction for demonstration
        title_start = html.find('<title>') + 7
        title_end = html.find('</title>')
        title = html[title_start:title_end] if title_start > 6 else "Unknown Title"
        
        prepared_data = {
            'source_domain': 'en.wikipedia.org',
            'page_title': title.replace(' - Wikipedia', ''),
            'scraped_bytes': len(html),
            'timestamp': time.time()
        }
        return prepared_data


## 6. Verification & Testing Block
Validating the fully upgraded pipeline. The execution loop dynamically initializes the database engine from the configuration and passes it into the scraping flow.

In [ ]:
print("========== FULL SCRAPE PIPELINE TEST WITH STRATEGY PATTERN ==========")

# 1. Setup the Database Strategy from Config
db_handler = DatabaseFactory.get_database(CONFIG['DB_TYPE'], CONFIG['DB_CONNECTION_STRING'])

# 2. Setup the Compliance Manager
manager = EthicalComplianceManager(CONFIG['USER_AGENT'])

target_url = 'https://en.wikipedia.org/wiki/Web_scraping'
target_collection = 'wikipedia_articles'

try:
    # Factory instantiates the exact scraper needed, injecting our handlers
    active_scraper = ScraperFactory.get_scraper(target_url, manager, db_handler)
    
    # Perform the scrape (handles politeness, fetch, parse, AND automated DB storage)
    active_scraper.scrape(target_url, destination=target_collection)
    
    print("\n[+] Scrape & Storage Pipeline completed successfully.")
    
except Exception as e:
    print(f"Pipeline Error: {e}")
